# TASK 1: Retinal Artery/Vein Segmentation using Color Fundus Images

## Objective

Develop a deep learning pipeline to perform pixel wise segmentation of retinal arteries and veins using only Color Fundus Photographs (CFP).

Input:
- Fundus Image (RGB)

Output:
- Segmentation Mask (RGB/Class Map)
  - Background = Black
  - Artery = Red
  - Vein = Blue
  - Crossing/Overlap = Green

---

# Problem Statement

Task 1 focuses on artery vein segmentation from CFP images only.

The model should learn:
- Vessel segmentation
- Artery vs vein classification
- Vessel crossing handling
- Thin vessel continuity preservation

Main challenges:
- Very small dataset (50 images)
- Severe class imbalance
- Thin vessel segmentation
- Artery and vein visual similarity
- Vessel crossings

---

# Dataset Information

Total Images: 50

Each sample contains:
- Fundus Image (Input)
- Ground Truth AV Mask (Label)

Task 1 uses:
- training/images
- training/av

Ignored for Task 1:
- FFA_A
- FFA_AV
- biomarkers
- validation folder

---

# Dataset Split

Training folder split manually into:

- Train = 35
- Validation = 7
- Test = 8

Future improvement:
- 5 Fold Cross Validation

---

# Ground Truth Label Encoding

RGB Mask → Class Labels

Class Mapping:
- Background (0,0,0) → Class 0
- Red (255,0,0) → Class 1
- Blue (0,0,255) → Class 2
- Green (0,255,0) → Class 3

Training target shape:
(H, W)

Model output shape:
(B, 4, H, W)

---

# Pipeline Overview

Fundus Image
→ Green Channel Extraction + CLAHE
→ Channel Reconstruction [R, CLAHE(G), B]
→ Random Patch Sampling
→ Data Augmentation
→ Normalization (ImageNet mean/std)
→ Swin UNet (pretrained encoder)
→ Segmentation Prediction
→ RGB Mask Generation
→ Evaluation (Val + Test)

---

# Step 1: Preprocessing

## Green Channel Extraction

Extract green channel from RGB image.

Reason:
Retinal vessels are best visible in green channel.

---

## CLAHE Enhancement

Apply CLAHE on the green channel only. Red and blue channels are left untouched.

Purpose:
- Improve vessel contrast
- Enhance thin vessels
- Reduce illumination variation

---

## Channel Reconstruction

Implementation:
[R, CLAHE(G), B]

Final shape:
(H, W, 3)

Reason:
Enhances vessel visibility via the green channel while preserving R and B, since artery/vein discrimination depends on subtle color differences between the two. A flattened [G,G,G] reconstruction was considered but rejected — it would discard the R/B color cues needed for AV classification, which is the core difficulty of this task.

---

## Normalization

After scaling to [0,1], images are normalized using ImageNet mean/std:

Mean: [0.485, 0.456, 0.406]
Std: [0.229, 0.224, 0.225]

Note:
Input is [R, CLAHE(G), B], not natural RGB, so ImageNet stats are an approximation rather than an exact match — used because the pretrained Swin encoder expects inputs in this normalization range.

---

# Step 2: Patch Based Training

Training is patch based.

Patch size:
512 × 512

Reason:
- Only 50 images available
- Patch extraction increases effective samples
- Better GPU memory management

---

## Patch Sampling Strategy

Random patch sampling from each image.

Patches per image:
- Train = 20
- Validation = 15

Approximate patches per epoch:
- Train = 700
- Validation = 105

---

## Patch Filtering

Discard patches with very low vessel content.

Condition:
Patch kept only if vessel ratio > 2%

Purpose:
Avoid excessive background learning.

---

# Step 3: Data Augmentation

Light augmentation only, train set only (validation/test use no augmentation).

Used:
- Horizontal Flip
- Vertical Flip
- Rotation ±15°
- Small Brightness/Contrast
- Small Gamma shift

Avoided:
- Blur
- Elastic transforms
- Strong distortion

Reason:
Retinal vessel geometry must remain realistic.

---

# Step 4: Model Architecture

Primary model:
## Swin UNet

Repository:
Official Swin-Unet implementation

Original repo limitations:
- Designed for Synapse CT segmentation
- Default input size = 224
- Default window size = 7

---

## Modifications for Task 1

### Input Resolution Changed

Original: 224 × 224
Modified: 512 × 512

---

### Number of Output Classes Changed

Original: 1000 or task specific
Modified: 4 classes (BG, Artery, Vein, Crossing)

---

### Window Size Changed

Original: window_size = 7

Problem:
At img_size=512, patch4 embedding produces feature maps of 128/64/32/16 across the 4 stages. None of these are divisible by 7, causing a shape mismatch during window partitioning.

Fix:
window_size = 8 (divides all four stage resolutions evenly)

---

## Final Swin Configuration

- Patch Size = 4
- Input Channels = 3
- Embed Dim = 96
- Depths = [2,2,2,2]
- Num Heads = [3,6,12,24]
- Window Size = 8

---

## Pretrained Weight Loading

Checkpoint: swin_tiny_patch4_window7_224.pth (ImageNet-1K pretrained, official Swin-Transformer release)

Loaded via the repo's `model.load_from(config)`.

Known limitation:
The checkpoint was trained with window_size=7; this model uses window_size=8. All `relative_position_bias_table`, `relative_position_index`, and `attn_mask` tensors are window-size-dependent and do not match shape, so `load_from` skips them by default. Patch embedding, QKV/projection weights, MLP weights, and LayerNorms still transfer correctly.

Fix applied:
A custom step interpolates the 8 encoder `relative_position_bias_table` tensors (bicubic interpolation from the 7-window grid to the 8-window grid) after `load_from`, recovering pretrained positional-attention knowledge for the encoder. `relative_position_index` and `attn_mask` are deterministic (computed from window size, not learned) and require no recovery. Decoder-side (`layers_up.*`) bias tables remain randomly initialized and are trained from scratch, same as the segmentation head.

---

# Step 5: Loss Function

Hybrid loss:

Loss = 0.3 × CE + 0.4 × Tversky + 0.3 × CLDice

---

## Weighted Cross Entropy

Purpose:
Pixel wise classification, weighted for class imbalance.

Class weights:
- Background = 0.3
- Artery = 1.5
- Vein = 1.5
- Crossing = 5.0

Note:
An earlier, more extreme weighting ([0.05, 1.5, 1.5, 7.0]) combined with a recall-favoring Tversky setting caused severe background→vessel over-prediction (high recall, low precision on artery/vein). Weights were rebalanced to reduce this while still prioritizing the rare crossing class.

---

## Tversky Loss

alpha = 0.5, beta = 0.5 (balanced — equal penalty on false positives and false negatives)

Note:
An earlier setting (alpha=0.3, beta=0.7) favored recall to help thin vessel detection, but combined with aggressive class weighting it pushed the model toward over-predicting vessel classes. Rebalanced to 0.5/0.5 as a precision/recall-neutral starting point.

---

## CLDice Loss

Purpose:
Preserve vessel topology and continuity.

Implementation detail:
Applied on binary vessel topology — Background vs (Artery + Vein + Crossing) — since it targets connectivity, not multiclass separation.

Known limitation:
CLDice's skeleton-overlap formulation does not directly penalize over-thick/blobby predictions, since a dilated false-positive mass can still overlap the ground-truth skeleton. Not yet isolated as a contributor to over-prediction behavior — flagged for future investigation, not yet acted on.

---

# Step 6: Optimizer

Optimizer: AdamW
- Learning Rate = 1e-4
- Weight Decay = 1e-4

Scheduler: ReduceLROnPlateau
- mode = max (tracks validation dice)
- factor = 0.5
- patience = 3

---

# Step 7: Training Configuration

Batch Size: 4
Epochs: 50 (current experimentation run; not yet the final configured run)
Mixed Precision: Enabled (torch.amp autocast + GradScaler)

---

# Step 8: Evaluation Metrics

Primary:
- Mean Dice Score
- IoU

Class wise:
- Artery / Vein / Crossing dice, precision, recall, specificity, accuracy, IoU

Background excluded from the mean dice used for checkpoint selection, but included in the full confusion-matrix-based metrics.

Evaluation is run on both:
- Validation set (used for checkpoint selection + scheduler)
- Held-out test set (final reporting only, never used for model selection)

---

# Step 9: Prediction Pipeline

Input Patch
→ Preprocessing (CLAHE + normalization)
→ Swin UNet
→ Logits
→ Softmax
→ Argmax
→ Class Mask
→ RGB Mask (0→Black, 1→Red, 2→Blue, 3→Green)

---

# Challenges Encountered During Implementation

## Issue 1: Swin UNet Repo Integration
Resolved using repo config with custom overrides (img_size, num_classes, window_size, pretrain checkpoint path).

## Issue 2: Window Partition Failure
512 input incompatible with window_size=7 (feature maps not divisible by 7). Fixed by setting window_size=8.

## Issue 3: Pretrained Weight Shape Mismatch
Changing window_size from 7→8 invalidates all relative position bias tables, indices, and attention masks in the checkpoint. `load_from` skips these automatically. Recovered the learned bias tables (encoder only) via bicubic interpolation; indices/masks require no recovery since they're deterministic, not learned.

## Issue 4: Windows DataLoader Workers Crash
num_workers > 0 caused worker crashes in VS Code Jupyter on Windows. Fixed with num_workers=0.

## Issue 5: Class Imbalance Overcorrection
Initial aggressive class weighting + recall-favoring Tversky loss caused severe background-to-vessel false positive rate. Diagnosed via confusion matrix (precision/recall gap) and corrected by rebalancing both class weights and Tversky alpha/beta together.

---

# Expected Deliverables

By end of Task 1:
- Stable Swin UNet training pipeline
- Best model checkpoint (selected via validation dice)
- Validation metrics (per-class + confusion matrix)
- Test set metrics (per-class + confusion matrix, held out from model selection)
- RGB segmentation masks
- Error analysis

---

# Future Experiments

- Attention UNet
- UNet++
- SegFormer
- Hybrid Architecture: CNN Encoder + Swin Bottleneck + Attention Decoder
- RRWNet preprocessing
- Interpolating decoder-side (layers_up) relative position bias tables, not just encoder
- Elastic/grid distortion augmentation given the very small source image count
- 100-epoch final training run once loss/weight configuration is validated at 50 epochs

---

# Goal

Build a complete end to end baseline for Task 1 using Swin UNet: from fundus image input to artery vein RGB segmentation output.

In [ ]:
import os
import cv2
import math
import time
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm import tqdm
from glob import glob

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from sklearn.metrics import confusion_matrix, classification_report

import albumentations as A

warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    SEED = 42
    
    IMAGE_SIZE = 512
    PATCH_SIZE = 512
    
    BATCH_SIZE = 4
    NUM_WORKERS = 0
    
    EPOCHS = 100
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    
    NUM_CLASSES = 4
    
    TRAIN_PATCHES_PER_IMAGE = 20
    VAL_PATCHES_PER_IMAGE = 15
    
    DEVICE = "cuda:0"
    
    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    SAVE_PATH = "./checkpoints"

In [ ]:
TRAIN_IMG_DIR = r"C:\Users\HP\Desktop\GAVE2_preliminary\training\images"
TRAIN_MASK_DIR = r"C:\Users\HP\Desktop\GAVE2_preliminary\training\av"

In [ ]:
assert torch.cuda.is_available(), "CUDA not available"

device = torch.device(CFG.DEVICE)

print("Using Device:", device)
print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)

In [ ]:
image_paths = sorted(glob(os.path.join(TRAIN_IMG_DIR, "*")))
mask_paths = sorted(glob(os.path.join(TRAIN_MASK_DIR, "*")))

print("Images:", len(image_paths))
print("Masks:", len(mask_paths))

In [ ]:
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.30,
    random_state=42
)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs,
    temp_masks,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

In [ ]:
def rgb_to_mask(mask):
    h, w, _ = mask.shape
    out = np.zeros((h, w), dtype=np.uint8)

    out[np.all(mask == [0, 0, 0], axis=-1)] = 0
    out[np.all(mask == [255, 0, 0], axis=-1)] = 1
    out[np.all(mask == [0, 0, 255], axis=-1)] = 2
    out[np.all(mask == [0, 255, 0], axis=-1)] = 3

    return out

In [ ]:
def preprocess_fundus(image):
    r = image[:, :, 0]
    g = image[:, :, 1]
    b = image[:, :, 2]

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g_enhanced = clahe.apply(g)

    processed = np.stack([r, g_enhanced, b], axis=-1)

    return processed

In [ ]:
sample = cv2.imread(train_imgs[0])
sample = cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)

processed = preprocess_fundus(sample)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(sample)
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(processed)
plt.title("Green + CLAHE")

plt.show()

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
])

In [ ]:
val_transform = A.Compose([])

In [ ]:
def valid_patch(mask, threshold=0.02):
    vessel_pixels = np.sum(mask > 0)
    total_pixels = mask.shape[0] * mask.shape[1]

    vessel_ratio = vessel_pixels / total_pixels
    return vessel_ratio > threshold

In [ ]:
def extract_random_patch(image, mask, patch_size=512):
    h, w = image.shape[:2]

    for _ in range(20):
        x = random.randint(0, w - patch_size)
        y = random.randint(0, h - patch_size)

        img_patch = image[y:y+patch_size, x:x+patch_size]
        mask_patch = mask[y:y+patch_size, x:x+patch_size]

        if valid_patch(mask_patch):
            return img_patch, mask_patch

    return img_patch, mask_patch

In [ ]:
class RetinalPatchDataset(Dataset):
    def __init__(
        self,
        image_paths,
        mask_paths,
        patches_per_image=20,
        transform=None
    ):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.patches_per_image = patches_per_image
        
        self.indices = []
        for idx in range(len(image_paths)):
            for _ in range(patches_per_image):
                self.indices.append(idx)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_idx = self.indices[idx]

        image = cv2.imread(self.image_paths[img_idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[img_idx])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

        mask = rgb_to_mask(mask)

        image = preprocess_fundus(image)

        image, mask = extract_random_patch(
            image,
            mask,
            CFG.PATCH_SIZE
        )

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        image = image.astype(np.float32) / 255.0
        image = (image - CFG.IMAGENET_MEAN) / CFG.IMAGENET_STD

        image = torch.tensor(image).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return image, mask

In [ ]:
train_dataset = RetinalPatchDataset(
    train_imgs,
    train_masks,
    patches_per_image=CFG.TRAIN_PATCHES_PER_IMAGE,
    transform=train_transform
)

val_dataset = RetinalPatchDataset(
    val_imgs,
    val_masks,
    patches_per_image=CFG.VAL_PATCHES_PER_IMAGE,
    transform=val_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)

In [ ]:
def mask_to_rgb(mask):
    rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    rgb[mask == 1] = [255, 0, 0]
    rgb[mask == 2] = [0, 0, 255]
    rgb[mask == 3] = [0, 255, 0]

    return rgb

In [ ]:
sample_img = images[0].permute(1, 2, 0).numpy()
sample_img = sample_img * CFG.IMAGENET_STD + CFG.IMAGENET_MEAN
sample_img = np.clip(sample_img, 0, 1)
sample_mask = masks[0].numpy()

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(sample_img)
plt.title("Input Patch")

plt.subplot(1,2,2)
plt.imshow(mask_to_rgb(sample_mask))
plt.title("GT Patch")

plt.show()

In [ ]:
import sys

repo_path = r"C:\Users\HP\Desktop\GAVE2_preliminary\Swin-Unet-main\Swin-Unet-main"
sys.path.append(repo_path)

In [ ]:
from config import get_config

In [ ]:
class Args:
    cfg = r"C:\Users\HP\Desktop\GAVE2_preliminary\Swin-Unet-main\Swin-Unet-main\configs\swin_tiny_patch4_window7_224_lite.yaml"
    
    opts = None
    batch_size = None
    zip = False
    cache_mode = None
    resume = None
    accumulation_steps = None
    use_checkpoint = False
    amp_opt_level = None
    tag = None
    eval = False
    throughput = False

In [ ]:
args = Args()
config = get_config(args)

In [ ]:
config.defrost()

config.DATA.IMG_SIZE = 512
config.MODEL.NUM_CLASSES = 4
config.MODEL.SWIN.IN_CHANS = 3
config.MODEL.SWIN.WINDOW_SIZE = 8
config.MODEL.PRETRAIN_CKPT = r"C:\Users\HP\Desktop\GAVE2_preliminary\Swin-Unet-main\Swin-Unet-main\swin_tiny_patch4_window7_224.pth"

config.freeze()

In [ ]:
from networks.vision_transformer import SwinUnet

In [ ]:
model = SwinUnet(
    config=config,
    img_size=512,
    num_classes=4
)

model = model.to(device)

In [ ]:
model.load_from(config)

In [ ]:
def interpolate_relative_position_bias(model, pretrained_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(pretrained_path, map_location=device)
    pretrained_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

    swin_dict = model.swin_unet.state_dict()
    updated = 0

    for k in list(swin_dict.keys()):
        if "layers." not in k or "relative_position_bias_table" not in k:
            continue
        if k not in pretrained_dict:
            continue

        pretrained_table = pretrained_dict[k]
        model_table = swin_dict[k]

        if pretrained_table.shape == model_table.shape:
            continue

        nH = pretrained_table.shape[1]
        old_size = int(pretrained_table.shape[0] ** 0.5)
        new_size = int(model_table.shape[0] ** 0.5)

        reshaped = pretrained_table.permute(1, 0).reshape(1, nH, old_size, old_size)
        resized = F.interpolate(
            reshaped, size=(new_size, new_size),
            mode='bicubic', align_corners=False
        )
        resized = resized.reshape(nH, new_size * new_size).permute(1, 0)

        swin_dict[k] = resized
        updated += 1

    model.swin_unet.load_state_dict(swin_dict)
    print(f"Interpolated {updated} encoder relative position bias tables")

interpolate_relative_position_bias(model, config.MODEL.PRETRAIN_CKPT)

In [ ]:
def soft_erode(img):
    if len(img.shape) == 4:
        p1 = -F.max_pool2d(-img, kernel_size=(3,1), stride=(1,1), padding=(1,0))
        p2 = -F.max_pool2d(-img, kernel_size=(1,3), stride=(1,1), padding=(0,1))
        return torch.min(p1, p2)

In [ ]:
def soft_dilate(img):
    return F.max_pool2d(img, kernel_size=3, stride=1, padding=1)

In [ ]:
def soft_open(img):
    return soft_dilate(soft_erode(img))

In [ ]:
def soft_skel(img, iter_=10):
    img1 = soft_open(img)
    skel = F.relu(img - img1)

    for _ in range(iter_):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)

    return skel

In [ ]:
class SoftCLDice(nn.Module):
    def __init__(self, iter_=10, smooth=1.):
        super().__init__()
        self.iter = iter_
        self.smooth = smooth

    def forward(self, preds, targets):
        probs = torch.softmax(preds, dim=1)

        vessel_prob = probs[:, 1:, :, :].sum(dim=1, keepdim=True)

        vessel_gt = (targets > 0).float().unsqueeze(1)

        skel_pred = soft_skel(vessel_prob, self.iter)
        skel_gt = soft_skel(vessel_gt, self.iter)

        tprec = (
            (torch.sum(skel_pred * vessel_gt) + self.smooth) /
            (torch.sum(skel_pred) + self.smooth)
        )

        tsens = (
            (torch.sum(skel_gt * vessel_prob) + self.smooth) /
            (torch.sum(skel_gt) + self.smooth)
        )

        cl_dice = 1. - 2.0 * (tprec * tsens) / (tprec + tsens)

        return cl_dice

In [ ]:
class_weights = torch.tensor([0.3, 1.5, 1.5, 5.0]).to(device)

ce_loss = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.softmax(preds, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=4).permute(0,3,1,2).float()

        dims = (0, 2, 3)

        TP = torch.sum(preds * targets_one_hot, dims)
        FP = torch.sum(preds * (1 - targets_one_hot), dims)
        FN = torch.sum((1 - preds) * targets_one_hot, dims)

        tversky = (TP + self.smooth) / (
            TP + self.alpha * FP + self.beta * FN + self.smooth
        )

        return 1 - tversky.mean()

In [ ]:
tversky_loss = TverskyLoss()
cldice_loss = SoftCLDice()

In [ ]:
def combined_loss(preds, targets):
    ce = ce_loss(preds, targets)
    tv = tversky_loss(preds, targets)
    cl = cldice_loss(preds, targets)

    loss = (
        0.3 * ce +
        0.4 * tv +
        0.3 * cl
    )

    return loss

In [ ]:
images, masks = next(iter(train_loader))
images = images.to(device)
masks = masks.to(device)

outputs = model(images)
loss = combined_loss(outputs, masks)

print(outputs.shape)
print(loss)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

scaler = GradScaler("cuda")

In [ ]:
def dice_score_per_class(preds, targets, num_classes=4, smooth=1e-6):
    preds = torch.argmax(preds, dim=1)

    scores = {}

    class_names = {
        1: "artery",
        2: "vein",
        3: "crossing"
    }

    dice_values = []

    for cls in range(1, num_classes):
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice = (2.0 * intersection + smooth) / (union + smooth)
        dice = dice.item()

        scores[class_names[cls]] = dice
        dice_values.append(dice)

    scores["mean_dice"] = np.mean(dice_values)

    return scores

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()

    running_loss = 0.0
    running_dice = 0.0

    loop = tqdm(loader, total=len(loader))

    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            outputs = model(images)
            loss = combined_loss(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        scores = dice_score_per_class(outputs, masks)
        dice = scores["mean_dice"]

        running_loss += loss.item()
        running_dice += dice

        loop.set_postfix(
            loss=loss.item(),
            dice=dice
        )

    epoch_loss = running_loss / len(loader)
    epoch_dice = running_dice / len(loader)

    return epoch_loss, epoch_dice

In [ ]:
def validate(model, loader):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    class_dice_totals = {"artery": 0.0, "vein": 0.0, "crossing": 0.0}

    with torch.no_grad():
        loop = tqdm(loader, total=len(loader))

        for images, masks in loop:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = combined_loss(outputs, masks)

            scores = dice_score_per_class(outputs, masks)
            dice = scores["mean_dice"]

            running_loss += loss.item()
            running_dice += dice
            for k in class_dice_totals:
                class_dice_totals[k] += scores[k]

            loop.set_postfix(loss=loss.item(), dice=dice)

    n = len(loader)
    epoch_loss = running_loss / n
    epoch_dice = running_dice / n
    per_class = {k: v / n for k, v in class_dice_totals.items()}

    return epoch_loss, epoch_dice, per_class

In [ ]:
best_dice = 0.0
EPOCHS = 50

In [ ]:
for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

    train_loss, train_dice = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    val_loss, val_dice, val_per_class = validate(
        model,
        val_loader
    )

    scheduler.step(val_dice)

    print(f"Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Dice:   {val_dice:.4f}")
    print(f"Val Dice per class: {val_per_class}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "best_swin_unet__nophoto_aug_task1.pth")
        print("Best model saved.")

In [ ]:
model.load_state_dict(torch.load("best_swin_unet_task1.pth"))
model.eval()

In [ ]:
images, masks = next(iter(val_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

preds = torch.argmax(outputs, dim=1).cpu().numpy()

In [ ]:
idx = 0

img = images[idx].cpu().permute(1,2,0).numpy()
img = img * CFG.IMAGENET_STD + CFG.IMAGENET_MEAN
img = np.clip(img, 0, 1)
gt = masks[idx].numpy()
pred = preds[idx]

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Input")

plt.subplot(1,3,2)
plt.imshow(mask_to_rgb(gt))
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(mask_to_rgb(pred))
plt.title("Prediction")

plt.show()

In [ ]:
all_preds = []
all_targets = []

model.eval()

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy().flatten())
        all_targets.extend(masks.cpu().numpy().flatten())

cm = confusion_matrix(all_targets, all_preds)
print(cm)

In [ ]:
test_dataset = RetinalPatchDataset(
    test_imgs,
    test_masks,
    patches_per_image=CFG.VAL_PATCHES_PER_IMAGE,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
test_preds = []
test_targets = []

model.eval()

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy().flatten())
        test_targets.extend(masks.cpu().numpy().flatten())

cm_test = confusion_matrix(test_targets, test_preds)
print(cm_test)

print(classification_report(
    test_targets,
    test_preds,
    target_names=["Background", "Artery", "Vein", "Crossing"],
    digits=4
))

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")


plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
def segmentation_metrics_from_cm(cm):
    num_classes = cm.shape[0]

    metrics = {}

    class_names = ["BG", "Artery", "Vein", "Crossing"]

    for i in range(num_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        precision = TP / (TP + FP + 1e-8)
        recall = TP / (TP + FN + 1e-8)
        specificity = TN / (TN + FP + 1e-8)
        accuracy = (TP + TN) / (cm.sum() + 1e-8)

        dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
        iou = TP / (TP + FP + FN + 1e-8)

        metrics[class_names[i]] = {
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "accuracy": accuracy,
            "dice": dice,
            "iou": iou
        }

    return metrics

In [ ]:
class_names = ["Background", "Artery", "Vein", "Crossing"]

print(classification_report(
    all_targets,
    all_preds,
    target_names=class_names,
    digits=4
))

In [ ]:
test_metrics = segmentation_metrics_from_cm(cm_test)

for cls, vals in test_metrics.items():
    print(f"\n{cls}")
    for k, v in vals.items():
        print(f"{k}: {v:.4f}")

In [ ]:
metrics = segmentation_metrics_from_cm(cm)

for cls, vals in metrics.items():
    print(f"\n{cls}")
    for k, v in vals.items():
        print(f"{k}: {v:.4f}")